<a href="https://colab.research.google.com/github/mixxr/gcolab-ml/blob/main/funnel/03_getting_real_data_chembl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Episode 3 — Getting Real Data (ChEMBL)
**Drug Discovery in Code** · [Life Sciences AI Applications]
To find real drugs we need real data: molecules that have actually been
tested against our target, and *how well they worked*. **ChEMBL** is the public database that has it —
millions of measured activities. We'll pull the known **EGFR** inhibitors and their potencies into a
pandas table, then convert potency into the scale everyone uses: **pIC50**.
> Educational only — not medical advice.

## Setup
`pip install chembl_webresource_client` (already in the repo env).

In [ ]:
pip install chembl_webresource_client

In [ ]:
from chembl_webresource_client.new_client import new_client
import pandas as pd, numpy as np

## 1. Find the target
Our target is **EGFR** — the epidermal growth factor receptor, ChEMBL id **CHEMBL203**.

In [ ]:
target = new_client.target
list(target.filter(target_chembl_id="CHEMBL203")
            .only(["pref_name", "organism"]))

## 2. Pull the measured activities
We ask for **IC50** values in **nM** with an exact (`=`) relation — the cleanest, most comparable
measurements. (EGFR has thousands; we cap at 1500 for a quick run — delete the cap to pull them all.)

In [ ]:
acts = new_client.activity.filter(
    target_chembl_id="CHEMBL203",
    standard_type="IC50",
    standard_relation="=",
    standard_units="nM",
).only(["molecule_chembl_id", "canonical_smiles", "standard_value"])

rows = []
for a in acts:
    if a["standard_value"] and a["canonical_smiles"]:
        rows.append((a["molecule_chembl_id"],
                     a["canonical_smiles"],
                     float(a["standard_value"])))
    if len(rows) >= 1500:   # cap for speed; remove to pull all
        break

df = pd.DataFrame(rows, columns=["molecule_chembl_id", "smiles", "IC50_nM"])
len(df)

## 3. Clean up, and convert IC50 → pIC50
A molecule can be measured many times, so we take the **median** per molecule. Then we convert
IC50 (in nM) to **pIC50** = 9 − log₁₀(IC50). Higher pIC50 = more potent; pIC50 of 6 means 1 µM,
9 means 1 nM. It's the scale every model in later episodes will predict.

In [ ]:
df = (df[df.IC50_nM > 0]
        .groupby(["molecule_chembl_id", "smiles"], as_index=False)
        .IC50_nM.median())

df["pIC50"] = 9 - np.log10(df.IC50_nM)
df = df.sort_values("pIC50", ascending=False).reset_index(drop=True)

print(len(df), "unique EGFR inhibitors")
df.head()[["molecule_chembl_id", "IC50_nM", "pIC50"]].round(2)

## 4. Look at the potency distribution

In [ ]:
df.pIC50.hist(bins=35)
import matplotlib.pyplot as plt
plt.axvline(6, color="orange", ls="--")   # 1 µM — a common activity cutoff
plt.xlabel("pIC50 (higher = more potent)"); plt.ylabel("compounds"); plt.show()

# save it — we'll reuse this dataset in every remaining episode
df.to_csv("./data/egfr_chembl.csv", index=False)

Just over a thousand real, measured EGFR inhibitors — from single-digit nanomolar blockbusters to
weak micromolar hits. **This dataset is the backbone of the rest of the series.**

## Recap
- **ChEMBL** gives you real molecules *and* how potent they were, for a target.
- We pulled EGFR IC50 data and tidied it into a pandas table.
- **pIC50** puts potency on a clean log scale — the label our models will predict.

**Next — Episode 4:** put these molecules through **drug-likeness filters** (Lipinski + structural
alerts) to trim the set down to the ones worth pursuing.